# BERT on WikiText — encoder activations in memory

Every transformer block outputs `(batch, seq_len, hidden)`, so a text run costs
far more per sample than a vision one. Downloads on first run: WikiText-2
(~5 MB) and BERT weights (~440 MB).

In [1]:
import sys
from pathlib import Path

# Make the repo-local examples._utils package importable when this notebook
# is opened directly from examples/text/, without installing anything extra.
sys.path.insert(0, str(Path.cwd().parents[1]))

import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer

from examples._utils.data import activation_loader
from examples._utils.text import WikiTextSamples
from nnact import ActivationPipeline
from nnact._model._hooked import HookedModel

MODEL = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
# A token-classification head gives per-token logits; its weights are
# randomly initialized on top of pretrained BERT, so predictions are
# meaningless here — only the encoder activations are of interest.
model = AutoModelForTokenClassification.from_pretrained(MODEL)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly

In [2]:
dataset = WikiTextSamples(tokenizer, n=256, max_length=64)
print(f"{len(dataset)} passages | input_ids {tuple(dataset[0]['input_ids'].shape)}")
print(f"real tokens in first: {int(dataset[0]['attention_mask'].sum())}")
print(dataset.texts[0][:90], "...")


256 passages | input_ids (64,)
real tokens in first: 64
Robert Boulter is an English film , television and theatre actor . He had a guest @-@ star ...


In [3]:
hooked = HookedModel(model)

# depth=3 reaches the individual blocks; depth=1 would only show
# "embeddings" and "encoder".
hooked.summary(depth=3).head(12)

,module,parameters
layer,,
bert,BertModel,108891648
bert.embeddings,BertEmbeddings,23837184
bert.embeddings.word_embeddings,Embedding,23440896
bert.embeddings.position_embeddings,Embedding,393216
bert.embeddings.token_type_embeddings,Embedding,1536
bert.embeddings.LayerNorm,LayerNorm,1536
bert.embeddings.dropout,Dropout,0
bert.encoder,BertEncoder,85054464
bert.encoder.layer,ModuleList,85054464


In [4]:
# Embeddings, an early block, a middle block, and the last one.
LAYERS = [
    "bert.embeddings",
    "bert.encoder.layer.0",
    "bert.encoder.layer.5",
    "bert.encoder.layer.11",
]

# run() accumulates every batch's activations in memory and hands back an
# ActivationDataset once the run finishes. Passing a tokenizer for a "token"
# pipeline also attaches token-level metadata: token_ids and decoded tokens.
pipeline = ActivationPipeline(model, LAYERS, output_type="token", tokenizer=tokenizer)
loader = activation_loader(dataset, batch_size=32)
activations = pipeline.run(loader)
activations.summary()

activations[1/8]  12%|#2         [00:00<?]

,sample,token_id,token,predicted_id,bert.embeddings_norm,bert.encoder.layer.0_norm,bert.encoder.layer.5_norm,bert.encoder.layer.11_norm
token,,,,,,,,
0,0,101,[CLS],1,14.468750,12.789062,20.500000,15.289062
1,0,2728,robert,0,14.679688,19.062500,21.734375,15.039062
2,0,8945,bo,0,16.640625,20.343750,22.156250,14.765625
3,0,11314,##ult,0,18.578125,20.234375,22.734375,15.382812
4,0,2121,##er,0,16.265625,18.687500,21.015625,13.335938
...,...,...,...,...,...,...,...,...
16148,255,2172,much,0,16.562500,20.093750,22.109375,16.406250
16149,255,2062,more,0,15.812500,19.671875,21.937500,15.390625
16150,255,8552,complicated,0,16.703125,20.015625,21.687500,15.742188


In [6]:
# Sanity check: the accumulated per-sample token ids must equal each
# passage's real (non-padding) input_ids, in order, and the decoded tokens
# must match what the tokenizer itself produces for those same ids.
for i in range(len(dataset)):
    mask = dataset[i]["attention_mask"].bool()
    expected_ids = dataset[i]["input_ids"][mask]

    sample = activations[i]
    assert (sample.token_ids == expected_ids.numpy()).all(), (
        f"sample {i}: token_ids do not match the original dataset"
    )
    assert sample.tokens.tolist() == tokenizer.convert_ids_to_tokens(
        expected_ids.tolist()
    ), f"sample {i}: decoded tokens do not match the tokenizer"

    # Recombining the accumulated tokens should round-trip back to (a
    # normalized form of) the original passage text. BERT's tokenizer
    # lowercases and strips accents/punctuation spacing, and the dataset
    # truncates to max_length, so compare against the tokenizer's own
    # decoding of the truncated ids rather than the raw source text. Special
    # tokens ([CLS]/[SEP]) are skipped on both sides, since decode() drops
    # them but convert_ids_to_tokens() does not.
    real_tokens = [
        token
        for token, token_id in zip(sample.tokens.tolist(), sample.token_ids.tolist())
        if token_id not in tokenizer.all_special_ids
    ]
    remapped = tokenizer.convert_tokens_to_string(real_tokens)
    expected_text = tokenizer.decode(expected_ids, skip_special_tokens=True)
    assert remapped.strip() == expected_text.strip(), (
        f"sample {i}: recombined tokens do not remap to the original text\n"
        f"  remapped: {remapped!r}\n"
        f"  expected: {expected_text!r}"
    )

print(f"token-level metadata verified for all {len(dataset)} samples")


token-level metadata verified for all 256 samples
